In [1]:
'''
task: FOL classification based on NL + CLIF text
model: t5-large
dataset: P-FOLIO
evaluation: supervised fine-tuning
'''
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# load FOLIO dataset
import pandas as pd

fol_train = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/p-folio/data/pfolio_train.csv")
fol_test = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/p-folio/data/pfolio_test.csv")
fol_valid = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/p-folio/data/pfolio_val.csv")

In [3]:
# transform data to acceptable format for model training
fol_train = fol_train.applymap(str)
fol_test = fol_test.applymap(str)
fol_valid = fol_valid.applymap(str)

/tmp/ipykernel_1901/3924469207.py:2: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  fol_train = fol_train.applymap(str)
/tmp/ipykernel_1901/3924469207.py:3: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  fol_test = fol_test.applymap(str)
/tmp/ipykernel_1901/3924469207.py:4: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  fol_valid = fol_valid.applymap(str)


In [4]:
# start preparing for QA pipeline
!pip install datasets
! pip install -U accelerate
! pip install -U transformers
!pip install transformers
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 29.3 MB/s eta 0:00:00
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.12.0
    Uninstalling accelerate-1.12.0:
      Successfully uninstalled accelerate-1.12.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 147.9 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.8 MB/s eta 0:00:00


In [5]:
import torch
import json
from tqdm import tqdm
import torch.nn as nn
from torch.optim import Adam
import nltk
import spacy
import string
import evaluate  # Bleu
from torch.utils.data import Dataset, DataLoader, RandomSampler
import pandas as pd
import numpy as np
import transformers
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from transformers import T5Tokenizer, T5Model, T5ForConditionalGeneration, T5TokenizerFast

import warnings
warnings.filterwarnings("ignore")

In [6]:
MODEL_NAME = "google/flan-t5-large"
TOKENIZER = T5TokenizerFast.from_pretrained(MODEL_NAME)
MODEL = T5ForConditionalGeneration.from_pretrained(MODEL_NAME, return_dict=True)
OPTIMIZER = Adam(MODEL.parameters(), lr=0.00001)
Q_LEN = 512   # Question Length
T_LEN = 512    # Target Length
BATCH_SIZE = 4
DEVICE = "cuda:0"

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [7]:
class QA_Dataset(Dataset):
    def __init__(self, tokenizer, dataframe, q_len, t_len):
        self.tokenizer = tokenizer
        self.q_len = q_len
        self.t_len = t_len
        self.data = dataframe
        self.subject = self.data["Premises - NL"] + " " + self.data["Premises - CLIF"]
        self.obj = self.data["Conclusions - NL"] + " " + self.data["Conclusions - CLIF"]
        self.relation = self.data['Truth Values']

    def __len__(self):
        return len(self.subject)

    def __getitem__(self, idx):
        subject = self.subject[idx]
        obj = self.obj[idx]
        relation = self.relation[idx]

        subject_tokenized = self.tokenizer(subject, obj, max_length=self.q_len, padding="max_length",
                                                    truncation=True, add_special_tokens=True)
        relation_tokenized = self.tokenizer(relation, max_length=self.t_len, padding="max_length",
                                          truncation=True, add_special_tokens=True)

        labels = torch.tensor(relation_tokenized["input_ids"], dtype=torch.long)
        labels[labels == 0] = -100

        return {
            "input_ids": torch.tensor(subject_tokenized["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(subject_tokenized["attention_mask"], dtype=torch.long),
            "labels": labels,
            "decoder_attention_mask": torch.tensor(relation_tokenized["attention_mask"], dtype=torch.long)
        }

In [8]:
train_dataset = QA_Dataset(TOKENIZER, fol_train, Q_LEN, T_LEN)
test_dataset = QA_Dataset(TOKENIZER, fol_valid, Q_LEN, T_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE)
val_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

In [9]:
#torch.cuda.empty_cache()
#del (MODEL)
MODEL.to('cuda')

T5ForConditionalGeneration(
  (shared): Embedding(32128, 1024)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 1024)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=1024, out_features=1024, bias=False)
              (k): Linear(in_features=1024, out_features=1024, bias=False)
              (v): Linear(in_features=1024, out_features=1024, bias=False)
              (o): Linear(in_features=1024, out_features=1024, bias=False)
              (relative_attention_bias): Embedding(32, 16)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=1024, out_features=2816, bias=False)
              (wi_1): Linear(in_features=1024, out_features=2816, bias=False)
       

In [10]:
train_loss = 0
val_loss = 0
train_batch_count = 0
val_batch_count = 0

for epoch in range(5):
    MODEL.train()
    for batch in tqdm(train_loader, desc="Training batches"):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        decoder_attention_mask = batch["decoder_attention_mask"].to(DEVICE)

        outputs = MODEL(
                          input_ids=input_ids,
                          attention_mask=attention_mask,
                          labels=labels,
                          decoder_attention_mask=decoder_attention_mask
                        )

        OPTIMIZER.zero_grad()
        outputs.loss.backward()
        OPTIMIZER.step()
        train_loss += outputs.loss.item()
        train_batch_count += 1

    #Evaluation
    MODEL.eval()
    for batch in tqdm(val_loader, desc="Validation batches"):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        decoder_attention_mask = batch["decoder_attention_mask"].to(DEVICE)

        outputs = MODEL(
                          input_ids=input_ids,
                          attention_mask=attention_mask,
                          labels=labels,
                          decoder_attention_mask=decoder_attention_mask
                        )

        OPTIMIZER.zero_grad()
        outputs.loss.backward()
        OPTIMIZER.step()
        val_loss += outputs.loss.item()
        val_batch_count += 1

    print(f"{epoch+1}/{5} -> Train loss: {train_loss / train_batch_count}\tValidation loss: {val_loss/val_batch_count}")

Validation batches: 100%|██████████| 24/24 [00:19<00:00,  1.26it/s]


1/5 -> Train loss: 3.8934189677238464	Validation loss: 2.2826214730739594


Validation batches: 100%|██████████| 24/24 [00:19<00:00,  1.26it/s]


2/5 -> Train loss: 2.5788277747730413	Validation loss: 1.4412562071035306


Validation batches: 100%|██████████| 24/24 [00:19<00:00,  1.26it/s]


3/5 -> Train loss: 1.9359463765113443	Validation loss: 1.1004780100451574


Validation batches: 100%|██████████| 24/24 [00:19<00:00,  1.26it/s]


4/5 -> Train loss: 1.590909605121447	Validation loss: 0.8912276258536925


Validation batches: 100%|██████████| 24/24 [00:19<00:00,  1.26it/s]

5/5 -> Train loss: 1.378771865947379	Validation loss: 0.7476285880431532


In [ ]:
MODEL.save_pretrained("/content/drive/MyDrive/Colab Notebooks/p-folio/models/flan-t5-large_nl_model")
TOKENIZER.save_pretrained("/content/drive/MyDrive/Colab Notebooks/p-folio/models/flan-t5-large_nl_tokenizer")

In [11]:
# evaluation metrics

import numpy as np
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report

def precision_at_k(actual, predicted):
    act_set = set(actual)
    pred_set = set(predicted)
    result = len(act_set & pred_set) / float(len(predicted))
    return result*100

def apk(actual, predicted, k):
    """
    Computes the average precision at k.
    This function computes the average prescision at k between two lists of
    items.
    Parameters
    ----------
    actual : list
             A list of elements that are to be predicted (order doesn't matter)
    predicted : list
                A list of predicted elements (order does matter)
    k : int
        The maximum number of predicted elements
    Returns
    -------
    score : double
            The average precision at k over the input lists
    """
    if not actual:
        return 0.0
    if len(predicted)>k:
        predicted = predicted[:k]
    score = 0.0
    num_hits = 0.0
    for i,p in enumerate(predicted):
        # first condition checks whether it is valid prediction
        # second condition checks if prediction is not repeated
        if p in actual and p not in predicted[:i]:
            num_hits += 1.0
            score += num_hits / (i+1.0)
    return score / min(len(actual), k)


def mapk(actual, predicted, k):
    """
    Computes the mean average precision at k.
    This function computes the mean average prescision at k between two lists
    of lists of items.
    Parameters
    ----------
    actual : list
             A list of lists of elements that are to be predicted
             (order doesn't matter in the lists)
    predicted : list
                A list of lists of predicted elements
                (order matters in the lists)
    k : int,
        The maximum number of predicted elements
    Returns
    -------
    score : double
            The mean average precision at k over the input lists
    """
    return np.mean([apk(a, p, k) for a,p in zip(actual, predicted)])


class EvaluationMetrics:

    def __init__(self, ks:list, metric="map") -> None:
        self.ks = ks
        self.metric=metric

    def evaluate(self, actual:list, predicted:list):
        if self.metric == "map":
            return self.MAP(actual, predicted)
        else:
            return self.AP(actual, predicted)

    def MAP(self, actual:list, predicted:list):
        results_dict = {}
        for k in self.ks:
            results_dict["MAP@"+str(k)] = mapk(actual=actual, predicted=predicted, k=k)
        return results_dict

    def AP(self, actual:list, predicted:list):
        results_dict = {}
        for k in self.ks:
            results_dict["AP@"+str(k)] = [apk(actual=actual, predicted=predicted, k=k)
                                          for a, p in zip(actual, predicted)]
        return results_dict

def predict_answer(obj, subject, ref_relation=None):
    inputs = TOKENIZER(subject, obj, return_tensors="pt").to(MODEL.device)

    outputs = MODEL.generate(input_ids=inputs["input_ids"], max_new_tokens=10)

    predicted_relation = TOKENIZER.batch_decode(outputs.detach().cpu().numpy(), skip_special_tokens=True)[0]

    print("premises: \n", subject)
    print("conclusion: \n", obj)
    print("true label: \n", ref_relation)
    print("predicted label: \n", predicted_relation)

    return predicted_relation

In [12]:
# test predictions
reference_labels = []
predicted_labels = []
for index, row in fol_test.iterrows():
  conclusion = row["Conclusions - NL"] + " " + row["Conclusions - CLIF"]
  premises = row["Premises - NL"] + " " + row["Premises - CLIF"]
  label = row["Truth Values"]

  predicted_label = predict_answer(conclusion, premises, label)
  reference_labels.append(label)
  predicted_labels.append(predicted_label)

# calculate classification metrics
print("Accuracy:", accuracy_score(reference_labels, predicted_labels))
print("Precision:", precision_score(reference_labels, predicted_labels, average="macro"))
print("Recall:", recall_score(reference_labels, predicted_labels, average="macro"))
print("F1:", f1_score(reference_labels, predicted_labels, average="macro"))
print("Classification Report:", classification_report(reference_labels, predicted_labels))

premises: 
 Tyga is a rapper.
Rappers release rap albums.
Tyga released the Well Done 3 album.
Rappers are not opera singers. israpper(tyga)
forall x forall y ((israpper(x) and releasedalbum(x, y)) implies israpalbum(y))
releasedalbum(tyga, welldone3)
forall x (israpper(x) implies not isoperasinger(x))
conclusion: 
 Well Done 3 is worth listening to. isworthlistening(welldone3)
true label: 
 U
predicted label: 
 U
premises: 
 Koei Tecmo is a Japanese video game and anime holding company.
Holding companies hold several companies.
Tecmo was disbanded in Japan, while Koei survived but was renamed.
Video game holding companies are holding companies. japanese(koeitecmo) and videogameholdingcompany(koeitecmo) and animeholdingcompany(koeitecmo) and holdingcompany(x)
forall x (holdingcompany(x) implies exists y(company(y) and holds(x, y)))
disbandsin(tecmo, japan) and survives(koei) and renames(koei)
forall x (videogameholdingcompany(x) implies holdingcompany(x))
conclusion: 
 Koei Tecmo holds